# El Framework Completo
### Del dato crudo a la decisión de negocio

`Fase 4 · Video 21` — serie **Forecasting de Ventas de la A a la Z** · **CIERRE**

Conectamos los 21 videos en un solo flujo: del CSV a la recomendación de inventario en pesos. Un
pipeline reutilizable que también potencia el dashboard en Streamlit.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.0f}M')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename, max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv('sales_history.csv')
sys.path.insert(0, str(csv_path.parents[1]))
from src.pipeline import run_framework

sales = pd.read_csv(csv_path, parse_dates=['date'])
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')
print(f'✅ {len(sales):,} filas | {sales["sku_id"].nunique()} SKUs | {catalog["demand_profile"].value_counts().to_dict()}')

---
## 2. El pipeline en una llamada

Todo el flujo — serie por SKU, pronóstico *tiered* (regular con estacional+deriva, intermitente con
Croston), reconciliación jerárquica, métricas y recomendación de inventario — se ejecuta en **una función**.
Planeamos el **último trimestre** (13 semanas) con nivel de servicio del 95%, y como tenemos el dato real,
medimos también qué tan bien lo hicimos.

In [ ]:
res = run_framework(sales, catalog, horizon=13, service_level=0.95)

print('RESUMEN DEL FRAMEWORK')
print('─' * 44)
print(f'  Horizonte planeado    : {res["horizon"]} semanas')
print(f'  Nivel de servicio     : {res["service_level"]:.0%}')
print(f'  WAPE (total)          : {res["metrics"]["wape"]:.1%}')
print(f'  BIAS (total)          : {res["metrics"]["bias"]:+.1%}')
print(f'  Jerarquía coherente   : {"✅ sí" if res["coherent"] else "❌ no"}')
print(f'  Inversión recomendada : ${res["total_investment"]:,.0f}')

---
## 3. Pronóstico del total vs. realidad

In [ ]:
total_fc = res['forecast'].sum(axis=1)
total_act = res['actual'].sum(axis=1)
hist = res['train'].sum(axis=1).iloc[-52:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hist.index, hist.values, color=BLUE, alpha=0.5, linewidth=1.2, label='histórico')
ax.plot(total_act.index, total_act.values, color='black', linewidth=2.5, label='real')
ax.plot(total_fc.index, total_fc.values, color=ORANGE, linewidth=2, linestyle='--', label='pronóstico')
ax.axvline(hist.index[-1], color='black', linestyle=':', linewidth=1)
ax.set_ylabel('unidades / semana')
ax.set_title('Pronóstico del total vs. lo que realmente pasó', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

---
## 4. Métricas y coherencia

El framework reporta la precisión con las métricas honestas del Video 18 y garantiza que los pronósticos de
SKU, categoría y total **cuadren** (reconciliación del Video 16).

In [ ]:
m = res['metrics']
print(f'  WAPE : {m["wape"]:.1%}   (magnitud del error)')
print(f'  BIAS : {m["bias"]:+.1%}   (dirección: +sobra / −falta)')
print(f'  MASE : {m["mase"]:.2f}   (vs seasonal naive)')
print(f'  Coherencia jerárquica: {"✅ SKU→categoría→total cuadran" if res["coherent"] else "❌"}')
print('\nNota honesta: el pronóstico por defecto es simple y robusto (estacional+deriva / Croston).')
print('Para exprimir precisión, se enchufa el SARIMAX (V12) o el XGBoost (V15) — ver sección 6.')

---
## 5. La decisión: recomendación de inventario en pesos

Aquí el forecasting se vuelve **negocio**. Para cada SKU:

`stock recomendado = pronóstico del horizonte + stock de seguridad`

donde el stock de seguridad depende del **nivel de servicio** y de la volatilidad de la demanda. Lo
traducimos a **pesos** con el precio de cada SKU y lo agregamos por tier.

In [ ]:
reco = res['reco']
by_tier = reco.groupby('tier').agg(skus=('reco_pesos', 'size'), pesos=('reco_pesos', 'sum'))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(by_tier.index, by_tier['pesos'], color=[GREEN, ORANGE])
axes[0].xaxis.set_major_formatter(money_fmt)
axes[0].set_title('Inversión recomendada por tier', fontsize=12, fontweight='bold')

top = reco.head(10).iloc[::-1]
axes[1].barh(top.index, top['reco_pesos'], color=BLUE)
axes[1].xaxis.set_major_formatter(money_fmt)
axes[1].set_title('Top 10 SKUs por inversión', fontsize=12, fontweight='bold')
fig.suptitle('Del pronóstico a la decisión de compra', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Inversión total recomendada: ${res["total_investment"]:,.0f}\n')
print(by_tier.assign(pesos=lambda d: d['pesos'].map('${:,.0f}'.format),
                     pct=lambda d: (d['skus'] / d['skus'].sum())).to_string())
print('\nEl patrón clásico de Supply Chain: los slow-movers (tier C) son MUCHOS SKUs pero POCO')
print('capital; el grueso de la inversión va a los regulares de alto volumen (tier A/B).')
print('\nMuestra de la recomendación por SKU:')
print(reco.head(5)[['tier', 'pred_units', 'safety_stock', 'reco_units', 'reco_pesos']].round(0).to_string())

---
## 6. Model-agnostic: enchufa tu mejor modelo

El framework separa **qué modelo pronostica** de **todo lo demás** (reconciliación, métricas, decisión). El
pronóstico por defecto (`forecast_sku`) es simple y robusto, pero puedes sustituirlo por cualquiera de la
serie —SARIMAX (V12), XGBoost global (V15), Chronos (V17)— sin tocar el resto del pipeline. Esa modularidad
es lo que hace un sistema mantenible en producción.

```python
# Ejemplo conceptual: reemplazar el forecaster por tu campeón
from src.pipeline import run_framework
# ... entrenas tu XGBoost/SARIMAX y generas forecast_df por SKU ...
# el resto (reconciliación, métricas, stock de seguridad, pesos) se reutiliza igual
```

---
## 7. Cierre de la serie

### El viaje completo (21 videos)

| Fase | Videos | Qué construimos |
|---|---|---|
| **1 · Fundamentos y EDA** | 1–7 | Dataset sintético, EDA, estacionariedad, descomposición, ACF/PACF, outliers, tiers |
| **2 · Feature Engineering** | 8–10 | Calendario (quincenas/festivos MX), lags/ventanas, precio/promoción |
| **3 · Modelado** | 11–17 | Baselines, ARIMA/SARIMAX, Prophet, intermitentes, XGBoost, jerárquico, foundation models |
| **4 · Evaluación y Producción** | 18–21 | Métricas, validación temporal, Optuna, **este framework** |

### Los principios que atraviesan todo

1. **Mide siempre contra un baseline** (Seasonal Naive). La complejidad se gana, no se asume.
2. **Valida en el tiempo**, nunca con KFold aleatorio; reporta distribución, no un número con suerte.
3. **La métrica correcta** (WAPE/BIAS/FVA), nunca MAPE con ceros.
4. **Coherencia jerárquica** para que el negocio confíe en los números.
5. **Traduce todo a pesos.** El forecasting no es un número: es capital, servicio y decisiones.

### El dashboard

Este mismo pipeline potencia `app.py` (Streamlit). Córrelo con:

```bash
pip install -r requirements-dev.txt
python main.py            # genera el dataset
streamlit run app.py      # abre el dashboard interactivo
```

---

**Gracias por acompañar la serie completa.** Deja de practicar con el dataset de pasajeros de 1949: ahora
tienes un pipeline realista, honesto y de punta a punta. 🚀